# 03 — Silver: Green Taxi

Reads Green Taxi records from Bronze (`tlc_bronze.green_raw`), applies data
quality rules via **flag-based annotation** (no silent drops), enriches with
zone metadata, builds the normalized Silver schema, and writes:
- **Valid records** → `tlc_silver.trips_green` (with `quality_flags` preserved)
- **Invalid records** → `tlc_audit.quarantine` (with `_rejection_reason`)

Silver `_meta` carries `bronze_run_id` + `source_file` for full lineage tracing.

> **Rules note:** Green uses `GREEN_RULES` = `YELLOW_GREEN_SHARED_RULES` (no datetime columns)
> + `GREEN_EXTRA_RULES` (`lpep_*` time order).  
> **Never** apply `YELLOW_RULES` or `YELLOW_GREEN_RULES` to Green data — they reference
> `tpep_*` columns that don't exist in Green DataFrames.

## Imports & Configuration

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from src.transformations.tlc_rules import (
    GREEN_RULES,                   # = YELLOW_GREEN_SHARED_RULES + GREEN_EXTRA_RULES
    apply_quality_flags,
    route_quarantine,
    print_rejection_summary,
)
from src.transformations.enrichment import load_zone_lookup, enrich_trip_locations
from src.transformations.schema import build_green_silver
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
from core.storage.mongo_client import get_audit_db
import pyspark.sql.functions as F
import datetime

YEARS_TO_PROCESS = [2023, 2024, 2025]
print("Imports OK.")

Imports OK.


## Start Spark & Audit Control

In [2]:
spark = get_spark(app_name="TLC_Silver_Green")

control = ControlManager("silver_green")
execution = control.start({"years": YEARS_TO_PROCESS, "vehicle_type": "green"})
run_id = execution.execution_id
print(f"Execution ID (Silver run_id): {run_id}")

2026-07-19 03:05:51 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
2026-07-19 03:05:51 | INFO     | tlc.audit.control_manager | [AUDIT] Execution started | pipeline=silver_green id=810eac9d
Execution ID (Silver run_id): 810eac9d


## Idempotency Purge

Ensures this notebook can be re-run safely without duplicating data by clearing previous artifacts for this pipeline.

In [ ]:
from core.storage.mongo_client import get_audit_db, get_silver_db
from core.config.settings import settings

client = get_silver_db()
# 1. Purge Silver collection for green
get_silver_db()["trips_green"].delete_many({})

# 2. Purge Quarantine records generated by this pipeline
get_audit_db()["quarantine"].delete_many({"pipeline": "silver_green"})

print("Idempotency purge complete for green. Safe to run.")


## Read from Bronze

In [3]:
df_bronze = read_mongo(spark, settings.MONGO_DB_BRONZE, "green_raw")

if "_meta" in df_bronze.columns:
    df_bronze = df_bronze.filter(F.year("lpep_pickup_datetime").isin(YEARS_TO_PROCESS))

# records_in will be evaluated after caching df_flagged


2026-07-19 03:05:53 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_bronze.green_raw


## Apply Data Quality Flags

`GREEN_RULES` contains:
- `valid_distance`, `valid_fare`, `valid_total` — numeric sanity (no datetime columns)
- `valid_pickup_zone`, `valid_dropoff_zone` — location ID in [1, 265]
- `valid_passengers` — passenger count in [1, 8]
- `valid_time_order_green` — `lpep_dropoff > lpep_pickup` (Green-specific)

In [4]:
df_flagged = apply_quality_flags(df_bronze, GREEN_RULES)
records_in = df_flagged.count()
print(f"Records read from Bronze: {records_in:,}")

valid_df, rejected_df = route_quarantine(df_flagged)

records_valid    = valid_df.count()
records_rejected = rejected_df.count()

print(f"Valid records   : {records_valid:,}")
print(f"Rejected records: {records_rejected:,}")
print(f"Quarantine rate : {records_rejected / records_in * 100:.2f}%" if records_in else "N/A")

print_rejection_summary(rejected_df)

control.log_quality_check(
    check_name="green_quality_flags",
    dataset=f"green_raw_years_{YEARS_TO_PROCESS}",
    records_checked=records_in,
    records_failed=records_rejected,
)

2026-07-19 03:05:53 | INFO     | tlc.transformations.tlc_rules | [RULES] Quality flags applied | rules=['valid_distance', 'valid_fare', 'valid_total', 'valid_pickup_zone', 'valid_dropoff_zone', 'valid_passengers', 'valid_vendor', 'valid_ratecode', 'valid_payment_type', 'valid_time_order_green', 'valid_trip_type_green', 'valid_pickup_year_green']
Records read from Bronze: 2,038,624
Valid records   : 1,789,811
Rejected records: 248,813
Quarantine rate : 12.20%
2026-07-19 03:06:01 | INFO     | tlc.transformations.tlc_rules | [RULES] Total records rejected: 248,813
2026-07-19 03:06:03 | INFO     | tlc.transformations.tlc_rules |          ↳ 'Trip type must be 1 (street-hail) or 2 (dispatch).': 129,995
2026-07-19 03:06:03 | INFO     | tlc.transformations.tlc_rules |          ↳ 'Trip distance must be greater than zero.': 89,796
2026-07-19 03:06:03 | INFO     | tlc.transformations.tlc_rules |          ↳ 'Passenger count must be between 1 and 8.': 21,188
2026-07-19 03:06:03 | INFO     | tlc.tra

QualityCheckResult(check_id='810eac9d_green_quality_flags', check_name='green_quality_flags', dataset='green_raw_years_[2023, 2024, 2025]', status=<QualityStatus.FAILED: 'FAILED'>, records_checked=2038624, records_passed=1789811, records_failed=248813, failure_rate=0.12204948043386127, details={})

## Route Rejected Records to Quarantine

In [5]:
if records_rejected > 0:
    # OPTIMIZATION: Distributed PySpark write instead of driver collect()
        seen_cols = set()
    raw_cols = []
    for c in rejected_df.columns:
        if c not in ("_rejection_reason", "quality_flags", "_meta") and c.lower() not in seen_cols:
            raw_cols.append(c)
            seen_cols.add(c.lower())
    
    quarantine_df_mapped = rejected_df.select(
        F.current_timestamp().alias("quarantined_at"),
        F.lit(run_id).alias("silver_run_id"),
        F.col("_meta.run_id").alias("bronze_run_id"),
        F.col("_meta.source_file").alias("source_file"),
        F.lit("silver_green").alias("pipeline"),
        F.col("_rejection_reason").alias("reason"),
        F.col("quality_flags").alias("quality_flags"),
        F.struct(*[F.col(c) for c in raw_cols]).alias("raw_record")
    )
    
    write_mongo(quarantine_df_mapped, settings.MONGO_DB_AUDIT, "quarantine", mode="append")
    print(f"Quarantined {records_rejected:,} records into tlc_audit.quarantine (Distributed)")
else:
    print("No records quarantined.")


2026-07-19 03:06:10 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_audit.quarantine (mode=append)
Quarantined 248,813 records into tlc_audit.quarantine (Distributed)


## Enrich with Zone Metadata

In [6]:
zone_df = load_zone_lookup(spark)
valid_df = enrich_trip_locations(valid_df, zone_df)
print("Zone enrichment complete.")

2026-07-19 03:06:10 | INFO     | tlc.transformations.enrichment | [ENRICH] Zone lookup loaded: 265 zones from /home/jovyan/work/data/raw/lookup/taxi_zone_lookup.csv
2026-07-19 03:06:10 | INFO     | tlc.transformations.enrichment | [ENRICH] Location IDs enriched with Borough and Zone names.
Zone enrichment complete.


## Build Silver Schema & Write to MongoDB

In [7]:
silver_df = build_green_silver(valid_df, run_id=run_id)
n_written = write_mongo(silver_df, settings.MONGO_DB_SILVER, "trips_green", mode="append", num_rows=records_valid)
print(f"Rows written to tlc_silver.trips_green: {n_written:,}")

2026-07-19 03:06:10 | INFO     | tlc.transformations.schema | [SCHEMA] Green Silver schema applied (with bronze lineage + quality_flags).
2026-07-19 03:06:29 | INFO     | tlc.spark_utils | [SPARK] Wrote 1,789,811 rows → MongoDB tlc_silver.trips_green (mode=append)
Rows written to tlc_silver.trips_green: 1,789,811


## Volumetric Traceability Check

In [8]:
print(f"╔══════════════════════════════════════════╗")
print(f"║  Volumetric Traceability Report          ║")
print(f"╠══════════════════════════════════════════╣")
print(f"║  Bronze records in  : {records_in:>16,}  ║")
print(f"║  Silver records out : {n_written:>16,}  ║")
print(f"║  Quarantine records : {records_rejected:>16,}  ║")
print(f"║  Delta (must be 0)  : {records_in - n_written - records_rejected:>16,}  ║")
print(f"╚══════════════════════════════════════════╝")
assert records_in == n_written + records_rejected, "DATA LOSS DETECTED — investigate immediately!"

control.end(
    ExecutionStatus.SUCCESS,
    {
        "records_read_from_bronze": records_in,
        "records_written_to_silver": n_written,
        "records_quarantined": records_rejected,
        "quarantine_rate_pct": round(records_rejected / records_in * 100, 4) if records_in else 0,
    },
)

╔══════════════════════════════════════════╗
║  Volumetric Traceability Report          ║
╠══════════════════════════════════════════╣
║  Bronze records in  :        2,038,624  ║
║  Silver records out :        1,789,811  ║
║  Quarantine records :          248,813  ║
║  Delta (must be 0)  :                0  ║
╚══════════════════════════════════════════╝
2026-07-19 03:06:29 | INFO     | tlc.audit.control_manager | [AUDIT] Execution finished | id=810eac9d status=SUCCESS duration=38.0s
2026-07-19 03:06:29 | INFO     | tlc.audit.control_manager | [AUDIT] Report saved → /home/jovyan/work/data/audit/executions/20260719_030629_810eac9d_silver_green.json
2026-07-19 03:06:29 | INFO     | tlc.audit.control_manager | [AUDIT] Report inserted into MongoDB tlc_audit.pipeline_runs (id=810eac9d)


ExecutionRecord(pipeline_name='silver_green', input_parameters={'years': [2023, 2024, 2025], 'vehicle_type': 'green'}, execution_id='810eac9d', start_time=datetime.datetime(2026, 7, 19, 3, 5, 51, 608366), end_time=datetime.datetime(2026, 7, 19, 3, 6, 29, 621956), status=<ExecutionStatus.SUCCESS: 'SUCCESS'>, output_summary={'records_read_from_bronze': 2038624, 'records_written_to_silver': 1789811, 'records_quarantined': 248813, 'quarantine_rate_pct': 12.2049}, quality_checks=[QualityCheckResult(check_id='810eac9d_green_quality_flags', check_name='green_quality_flags', dataset='green_raw_years_[2023, 2024, 2025]', status=<QualityStatus.FAILED: 'FAILED'>, records_checked=2038624, records_passed=1789811, records_failed=248813, failure_rate=0.12204948043386127, details={})], errors=[])

## Audit Report

In [9]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))

{
  "execution_id": "810eac9d",
  "pipeline_name": "silver_green",
  "status": "SUCCESS",
  "start_time": "2026-07-19T03:05:51.608366",
  "end_time": "2026-07-19T03:06:29.621956",
  "duration_seconds": 38.01,
  "input_parameters": {
    "years": [
      2023,
      2024,
      2025
    ],
    "vehicle_type": "green"
  },
  "output_summary": {
    "records_read_from_bronze": 2038624,
    "records_written_to_silver": 1789811,
    "records_quarantined": 248813,
    "quarantine_rate_pct": 12.2049
  },
  "quality_checks": [
    {
      "check_id": "810eac9d_green_quality_flags",
      "check_name": "green_quality_flags",
      "dataset": "green_raw_years_[2023, 2024, 2025]",
      "status": "FAILED",
      "records_checked": 2038624,
      "records_passed": 1789811,
      "records_failed": 248813,
      "failure_rate": 0.122049,
      "details": {}
    }
  ],
  "quality_passed": 0,
  "errors": []
}
